# Lab 3 - Four Estimators, One Question

*SDAIA Academy · Experimentation and Causal Inference · STARTER notebook*

## Objective
Estimate the same treatment effect four ways on one observational dataset with known ground truth, add a quasi-experimental design on a second dataset, and reconcile the spread by naming the assumption each estimate relies on.

Question: **Did the voluntary digital-literacy training programme raise service completion?** (`injaz_users.csv`, treatment `enrolled`, outcome `completed`). Enrolment was self-selected: the naive gap overstates the effect.

Second dataset: `injaz_regions_panel.csv`, the staggered region-by-region **reminder rollout** (DiD).

In [ ]:
import sys; sys.path.insert(0, '..')   # so `causal_utils` is importable from starter/ or solution/
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy import stats
import statsmodels.formula.api as smf
from causal_utils import *
plt.rcParams['figure.figsize'] = (7, 3.5)
rng = np.random.default_rng(213)
DATA = '../data'

In [ ]:
obs = pd.read_csv(f'{DATA}/injaz_users.csv')
naive = obs.loc[obs.enrolled==1,'completed'].mean() - obs.loc[obs.enrolled==0,'completed'].mean()
print(f'Naive difference (enrolled - not): {naive:+.4f}')
obs.head()

## Step 1 - Draw the DAG and select the adjustment set
Assumed structure: `age`, `digital_literacy`, `prior_completions`, `region_code` are common causes of `enrolled` and `completed`. `support_calls` is caused by `enrolled` and affects `completed` (a **mediator**). `distance_km` affects `online_filing` only.

List the backdoor paths with networkx and state the adjustment set. `support_calls` must be excluded (descendant of treatment).

In [ ]:
import networkx as nx
# TODO: build the DiGraph from the structure described above, list backdoor paths, and define ADJ (the adjustment set)
ADJ = [...]

## Step 2 - Outcome regression (g-formula)
Fit `E[Y | T, X]`, predict both potential outcomes for everyone, average the difference. Also fit the **wrong** model that adds the mediator, to see the effect shrink.

In [ ]:
f_adj = 'completed ~ enrolled + age + digital_literacy + prior_completions + C(region_code)'
m = smf.ols(f_adj, data=obs).fit(cov_type='HC3')
# TODO: predict with enrolled=1 and enrolled=0 for everyone (df.assign), average the difference -> g_ate
g_ate = ...
# TODO: refit with '+ support_calls' and compare the coefficient on enrolled

## Step 3 - Propensity score, overlap, stabilised IPW with trimming
Fit `e(x)` with logistic regression, plot the overlap, print the max and 99th-percentile weight, trim to common support, and check post-weighting balance (SMD < 0.1).

In [ ]:
from sklearn.linear_model import LogisticRegression
X = pd.get_dummies(obs[ADJ], columns=['region_code'], drop_first=True).astype(float)
ps = LogisticRegression(max_iter=2000).fit(X, obs['enrolled'])
obs['pscore'] = ps.predict_proba(X)[:, 1]
# TODO: histogram of pscore by arm (overlap plot)
# TODO: trim to common support (1st pct of treated .. 99th pct of control), call ipw_ate(..., stabilised=True)
# TODO: print max and 99th percentile weight; balance_table(..., weights='w')

## Step 4 - Doubly-robust AIPW with cross-fitting
Use gradient boosting for both nuisance models, fit on one fold and predict on the other (2-fold cross-fitting), then combine with `aipw_ate`. Report the interval.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import KFold
y, t = cs['completed'].values, cs['enrolled'].values
Xc = pd.get_dummies(cs[ADJ], columns=['region_code'], drop_first=True).astype(float).values
e_hat = np.zeros(len(y)); m1 = np.zeros(len(y)); m0 = np.zeros(len(y))
for train, test in KFold(2, shuffle=True, random_state=0).split(Xc):
    # TODO: fit propensity model on train, predict e_hat on test
    # TODO: fit outcome model on [t, X] train; predict m1 (t=1) and m0 (t=0) on test
    ...
dr = aipw_ate(y, t, e_hat, m1, m0)
print(dr)

## Step 5 - Quasi-experiment: difference-in-differences on the reminder rollout
Regions were activated in three waves (months 10, 14, 18); two regions were never treated. Estimate the TWFE DiD with region-clustered SEs, then the event study, then a placebo (shift activation 6 months earlier, pre-period only).

In [ ]:
panel = pd.read_csv(f'{DATA}/injaz_regions_panel.csv')
# TODO: naive before/after in treated regions
# TODO: TWFE DiD: smf.ols('completion_rate ~ treated_post + C(region) + C(month)').fit(cov_type='cluster', cov_kwds={'groups': panel['region']})
# TODO: event study with rel = month - activation_month (reference -1), plot coefficients with CIs
# TODO: placebo: keep pre-activation rows only, fake_post = month >= activation_month - 6

## Step 6 - Tabulate all estimates against ground truth and explain each gap
The instructor file `data/ground_truth.json` holds the planted effects. Fill the reconciliation table, then write one sentence per row naming the assumption it relies on.

In [ ]:
import json
truth = json.load(open(f'{DATA}/ground_truth.json'))   # ask the instructor to release this file at the end of the lab
# TODO: build the comparison table: naive, g-formula, IPW, AIPW, mediator-controlled, with truth and gap; add the DiD row

## Step 7 - Written reconciliation
For each method write one sentence: which assumption it leans on, and why its number sits where it does relative to the truth. Then answer: which single unmeasured variable would break the adjustment-based estimates, and what would you do about it?

*Your answer:*

...

### Fast finishers
1. Sensitivity: compute the E-value for the AIPW risk ratio (`e_value(rr)` in causal_utils).
2. Collider demo: generate two independent traits and a collider `completed = skill + effort + noise > 1`; show the correlation between the traits goes from 0 to negative once you condition on `completed == 1`.
3. IV: use `distance_km` as an instrument for `online_filing` (2SLS via `linearmodels` if installed, otherwise the Wald ratio on a binarised instrument). Report the first-stage F.